# Обробка даних маршрутів у Pandas
## Читання та запис CSV, очищення й аналіз датасету

**Тема:** аналіз датасету туристичних маршрутів (частина проєкту «Tourist Routes Manager»).

**Мета:** навчитися зчитувати, очищати та аналізувати табличні дані за допомогою Pandas, застосовувати фільтрацію, групування та статистичний аналіз, формувати нові датасети.

**План:** зчитування → первинне ознайомлення → очищення → приведення типів → нові стовпці → дослідницький аналіз → фільтрація й сортування → групування → збереження результатів.

## 1. Вибір і опис датасету

Для роботи використано датасет **`data/raw/routes_dataset.csv`** — колекцію туристичних маршрутів, структура якої повторює модель даних з основного коду (`Route`, `HikingRoute`, `CyclingRoute`, `BusTour`).
Датасет згенеровано автоматично модулем `tourist_routes/analytics/generate_dataset.py` (NumPy, `seed=42`, тож результат відтворюваний). У нього навмисно додано типові дефекти реальних даних: пропуски, повні дублікати, зайві пробіли й нестабільний регістр, число з комою замість крапки, від'ємні й завеликі значення, некоректні дати та оцінки поза діапазоном 1–10.

**Про що датасет:** опис пішохідних, велосипедних, автобусних та оглядових маршрутів у різних країнах: довжина, відгуки туристів, складність, ціна тощо.

**Важливі дані:** тип маршруту, країна, довжина (км), відгуки (оцінки), ціна.

| Стовпець | Значення |
|---|---|
| `route_id` | Ідентифікатор маршруту |
| `type` | Тип маршруту: `Route`, `HikingRoute`, `CyclingRoute`, `BusTour` |
| `name` | Назва маршруту |
| `country` | Країна |
| `start_city` | Місто старту |
| `length_km` | Протяжність маршруту, км |
| `description` | Короткий опис |
| `ratings` | Відгуки туристів — оцінки 1–10 через пробіл (як вводяться в консольному застосунку) |
| `difficulty` | Складність (лише для `HikingRoute`) |
| `elevation_gain_m` | Набір висоти, м (лише для `HikingRoute`) |
| `bike_type` | Тип велосипеда (лише для `CyclingRoute`) |
| `stops_count` | Кількість зупинок (лише для `BusTour`) |
| `date_added` | Дата додавання маршруту до каталогу |
| `price_eur` | Вартість участі, євро |

## 2. Підготовка середовища та зчитування даних

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Шукаємо датасет у структурі проєкту (notebooks/ -> data/raw/), інакше — поруч із ноутбуком (Colab)
CANDIDATES = [Path("../data/raw/routes_dataset.csv"), Path("data/raw/routes_dataset.csv"), Path("routes_dataset.csv")]
DATA_FILE = next((p for p in CANDIDATES if p.exists()), Path("routes_dataset.csv"))
OUT_DIR = Path("../data/processed") if Path("../data").exists() else Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, os.path.abspath(".."))   # щоб імпортувати tourist_routes (основного коду)

# --- Google Colab: завантаження файлу вручну (в Jupyter цей блок пропускається) ---
try:
    from google.colab import files
    if not DATA_FILE.exists():
        print("Оберіть файл routes_dataset.csv і дочекайтесь завантаження...")
        files.upload()
except ImportError:
    pass

# Якщо файлу немає — генеруємо його генератором із проєкту
if not DATA_FILE.exists():
    from tourist_routes.analytics.generate_dataset import generate
    generate(str(DATA_FILE))

In [2]:
df = pd.read_csv(DATA_FILE)
print("Датасет успішно зчитано.")

Датасет успішно зчитано.


### 2.1. Перші та останні рядки

In [3]:
df.head()

,route_id,type,name,country,start_city,length_km,description,ratings,difficulty,elevation_gain_m,bike_type,stops_count,date_added,price_eur
0,229,BusTour,Екскурсійний тур Балтики (Пассау),АВСТРІЯ,Пассау,282.6,Маршрут проходить через національний парк.,8 8 8,NaN,NaN,NaN,3.0,2025-08-17,111.95
1,10,CyclingRoute,Веломаршрут Фйордів (Шамоні),Франція,Шамоні,191.4,Мальовничий маршрут з чудовими краєвидами.,5 6 7 8 9 5 7,NaN,NaN,гірський,NaN,2023-03-19,44.56
2,58,CyclingRoute,Веломаршрут Доломітів (Відень),Австрія,Відень,293.4,Мальовничий маршрут з чудовими краєвидами.,NaN,NaN,NaN,гірський,NaN,2025-06-24,248.80
3,61,BusTour,Екскурсійний тур Карпат (Одеса),Україна,Одеса,423.3,Є кемпінги та місця для відпочинку по дорозі.,9 9 9 7,NaN,NaN,NaN,10.0,2024-09-09,347.18
4,26,HikingRoute,Стежка Карпат (Пассау),Австрія,Пассау,6.3,NaN,8 7 7 8 6 6 8,складний,2252.0,NaN,NaN,2025-11-21,-10.00


In [4]:
df.tail()

,route_id,type,name,country,start_city,length_km,description,ratings,difficulty,elevation_gain_m,bike_type,stops_count,date_added,price_eur
307,189,BusTour,автобусом через дністра (київ),Україна,Київ,204.4,Є кемпінги та місця для відпочинку по дорозі.,9 10 8,NaN,NaN,NaN,6.0,2023-12-18,278.70
308,72,CyclingRoute,Веломаршрут Фйордів (Ставангер),Норвегія,Ставангер,140.1,Підходить для підготовлених туристів.,10 9 10 10,NaN,NaN,гірський,NaN,2025-08-30,NaN
309,107,CyclingRoute,Трейл Татр (Одеса),Україна,Одеса,160.7,Популярний маршрут у літній сезон.,8 5 5,NaN,NaN,шосейний,NaN,2023-03-03,460.42
310,271,HikingRoute,Піший похід Альп (Братислава),Словаччина,Братислава,19.0,Зручний варіант для вихідних.,6 9,складний,1360.0,NaN,NaN,2024-09-29,534.86
311,103,HikingRoute,Трек через Озерного краю (Берген),Норвегія,Берген,50.1,Маршрут проходить через національний парк.,10 10 10 7,легкий,517.0,NaN,NaN,2024-10-09,145.81


### 2.2. Розмір, стовпці, типи даних

In [5]:
print("Кількість рядків і стовпців:", df.shape)
print("Рядків:", df.shape[0], "| Стовпців:", df.shape[1])
print("\nНазви стовпців:")
print(list(df.columns))

Кількість рядків і стовпців: (312, 14)
Рядків: 312 | Стовпців: 14

Назви стовпців:
['route_id', 'type', 'name', 'country', 'start_city', 'length_km', 'description', 'ratings', 'difficulty', 'elevation_gain_m', 'bike_type', 'stops_count', 'date_added', 'price_eur']


In [6]:
df.dtypes

route_id              int64
type                    str
name                    str
country                 str
start_city              str
length_km               str
description             str
ratings                 str
difficulty              str
elevation_gain_m    float64
bike_type               str
stops_count         float64
date_added              str
price_eur           float64
dtype: object

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 312 entries, 0 to 311
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   route_id          312 non-null    int64  
 1   type              312 non-null    str    
 2   name              312 non-null    str    
 3   country           312 non-null    str    
 4   start_city        312 non-null    str    
 5   length_km         306 non-null    str    
 6   description       302 non-null    str    
 7   ratings           304 non-null    str    
 8   difficulty        117 non-null    str    
 9   elevation_gain_m  117 non-null    float64
 10  bike_type         75 non-null     str    
 11  stops_count       63 non-null     float64
 12  date_added        312 non-null    str    
 13  price_eur         296 non-null    float64
dtypes: float64(3), int64(1), str(10)
memory usage: 34.3 KB


### 2.3. Кількість унікальних значень

In [8]:
df.nunique()

route_id            300
type                  4
name                293
country              15
start_city           33
length_km           266
description           6
ratings             258
difficulty            3
elevation_gain_m    106
bike_type             3
stops_count          13
date_added          268
price_eur           282
dtype: int64

**Спостереження.** `length_km` та `price_eur` мають тип `object` (рядок), хоча мають бути числами — у стовпці є значення з комою («12,5»). Значення `stops_count` збережено як `float` через пропуски. Різні варіанти запису країн (`nunique` для `country` більше за реальну кількість країн) вказують на проблеми з регістром і пробілами.

## 3. Очищення даних

### 3.1. Пропущені значення

In [9]:
missing = df.isnull().sum()
missing_percent = (df.isnull().mean() * 100).round(1)
pd.DataFrame({"пропусків": missing, "%": missing_percent})

,пропусків,%
route_id,0,0.0
type,0,0.0
name,0,0.0
country,0,0.0
start_city,0,0.0
length_km,6,1.9
description,10,3.2
ratings,8,2.6
difficulty,195,62.5
elevation_gain_m,195,62.5


Частина пропусків **структурна**: `difficulty` та `elevation_gain_m` існують лише для пішохідних маршрутів, `bike_type` — для велосипедних, `stops_count` — для автобусних турів. Їх **не заповнюємо**, бо значення просто не застосовне до інших типів. Перевіримо це:

In [10]:
specific = {
    "difficulty": "HikingRoute",
    "elevation_gain_m": "HikingRoute",
    "bike_type": "CyclingRoute",
    "stops_count": "BusTour",
}
for column, route_type in specific.items():
    wrong = df[(df["type"] != route_type) & df[column].notna()]
    empty_for_type = df[(df["type"] == route_type) & df[column].isna()]
    print(f"{column:<17} значення у чужих типах: {len(wrong)}, пропусків у «своєму» типі: {len(empty_for_type)}")

difficulty        значення у чужих типах: 0, пропусків у «своєму» типі: 0
elevation_gain_m  значення у чужих типах: 0, пропусків у «своєму» типі: 0
bike_type         значення у чужих типах: 0, пропусків у «своєму» типі: 0
stops_count       значення у чужих типах: 0, пропусків у «своєму» типі: 0


### 3.2. Дублікати

In [11]:
print("Повних дублікатів:", df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values("route_id").head(6)

Повних дублікатів: 12


,route_id,type,name,country,start_city,length_km,description,ratings,difficulty,elevation_gain_m,bike_type,stops_count,date_added,price_eur
246,2,CyclingRoute,Велотур Балтики (Ліон),Франція,Ліон,249.4,Маршрут проходить через національний парк.,9 8 10 8 9 8 8,NaN,NaN,шосейний,NaN,2023-03-21,NaN
263,2,CyclingRoute,Велотур Балтики (Ліон),Франція,Ліон,249.4,Маршрут проходить через національний парк.,9 8 10 8 9 8 8,NaN,NaN,шосейний,NaN,2023-03-21,NaN
212,5,HikingRoute,Хребтами Дунаю (Одда),Норвегія,Одда,11.6,Мальовничий маршрут з чудовими краєвидами.,9 9 9 7 8 8,легкий,2814.0,NaN,NaN,2024-01-03,105.71
146,5,HikingRoute,Хребтами Дунаю (Одда),Норвегія,Одда,11.6,Мальовничий маршрут з чудовими краєвидами.,9 9 9 7 8 8,легкий,2814.0,NaN,NaN,2024-01-03,105.71
170,13,BusTour,Автотур Монблану (Рим),Італія,Рим,340.0,Популярний маршрут у літній сезон.,9 7 8 7,NaN,NaN,NaN,9.0,2024-06-05,395.81
77,13,BusTour,Автотур Монблану (Рим),Італія,Рим,340.0,Популярний маршрут у літній сезон.,9 7 8 7,NaN,NaN,NaN,9.0,2024-06-05,395.81


Це повні копії рядків (однаковий `route_id` і всі поля), тому їх безпечно видалити.

In [12]:
rows_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Видалено дублікатів: {rows_before - len(df)}. Залишилось рядків: {len(df)}")
print("route_id унікальний:", df["route_id"].is_unique)

Видалено дублікатів: 12. Залишилось рядків: 300
route_id унікальний: True


### 3.3. Перевірка та виправлення текстових значень

In [13]:
print("Країни до очищення:")
print(sorted(df["country"].unique()))

Країни до очищення:
['  ІТАЛІЯ ', '  АВСТРІЯ ', '  НОРВЕГІЯ ', '  ПОЛЬЩА ', '  СЛОВАЧЧИНА ', '  ФРАНЦІЯ ', 'Італія', 'Австрія', 'Норвегія', 'Перу', 'Польща', 'Словаччина', 'Україна', 'Франція', 'Швейцарія']


In [14]:
def capitalize_first(series):
    # Перша літера велика, решта регістру не змінюється
    series = series.str.strip()
    return series.str[:1].str.upper() + series.str[1:]

df["country"] = df["country"].str.strip().str.capitalize()
df["start_city"] = df["start_city"].str.strip().str.capitalize()
df["name"] = capitalize_first(df["name"])
df["description"] = df["description"].str.strip()

print("Країни після очищення:")
print(sorted(df["country"].unique()))
print("Унікальних країн:", df["country"].nunique())

Країни після очищення:
['Італія', 'Австрія', 'Норвегія', 'Перу', 'Польща', 'Словаччина', 'Україна', 'Франція', 'Швейцарія']
Унікальних країн: 9


In [15]:
# Перевірка категоріальних значень: допустимі лише значення з config.py
VALID_TYPES = {"Route", "HikingRoute", "CyclingRoute", "BusTour"}
VALID_DIFFICULTY = {"легкий", "середній", "складний"}
VALID_BIKES = {"шосейний", "гірський", "міський"}

print("Невідомі типи:", set(df["type"].unique()) - VALID_TYPES)
print("Невідома складність:", set(df["difficulty"].dropna().unique()) - VALID_DIFFICULTY)
print("Невідомі типи велосипеда:", set(df["bike_type"].dropna().unique()) - VALID_BIKES)
print("Порожні назви:", (df["name"].str.len() == 0).sum())

Невідомі типи: set()
Невідома складність: set()
Невідомі типи велосипеда: set()
Порожні назви: 0


### 3.4. Числові дані: приведення до числового типу та перевірка коректності

У `length_km` трапляється кома як десятковий роздільник. Замінюємо її на крапку й перетворюємо на число (`errors="coerce"` перетворює непридатні значення на `NaN`).

In [16]:
for column in ["length_km", "price_eur"]:
    df[column] = pd.to_numeric(
        df[column].astype(str).str.replace(",", ".", regex=False), errors="coerce"
    )
df[["length_km", "price_eur"]].dtypes

length_km    float64
price_eur    float64
dtype: object

In [17]:
# Межі коректних значень взято з config.py: довжина 0.1 – 40000 км
MIN_LENGTH_KM, MAX_LENGTH_KM = 0.1, 40000

bad_length = df[(df["length_km"] < MIN_LENGTH_KM) | (df["length_km"] > MAX_LENGTH_KM)]
bad_price = df[df["price_eur"] < 0]
print("Некоректна довжина:", len(bad_length), "| Від'ємна ціна:", len(bad_price))
bad_length[["route_id", "type", "name", "length_km"]]

Некоректна довжина: 6 | Від'ємна ціна: 3


,route_id,type,name,length_km
5,64,BusTour,Автобусом через Дністра (Кошице),-25.0
89,20,BusTour,Автотур Монблану (Болцано),99999.0
150,121,CyclingRoute,Веломаршрут Озерного краю (Страсбург),99999.0
172,143,Route,Оглядова екскурсія Монблану (Шамоні),-25.0
213,163,HikingRoute,Стежка Альп (Попрад),-25.0
223,275,BusTour,Автотур Дністра (Ставангер),99999.0


### 3.5. Обробка пропусків і некоректних значень

Прийняті рішення:

| Проблема | Рішення | Обґрунтування |
|---|---|---|
| Некоректна або відсутня `length_km` | видалити рядок | довжина — ключовий показник, вигадувати її не можна |
| Від'ємна `price_eur` | вважати пропуском | ціна не може бути від'ємною |
| Пропуски `price_eur` | заповнити медіаною за типом маршруту | ціни різних типів сильно відрізняються, медіана стійка до викидів |
| Пропуски `description` | замінити на «Опис відсутній» | це текстове поле, не показник |
| Пропуски `ratings` | залишити | маршрут без відгуків — норма (у основного коду середній бал таких маршрутів `None`) |

In [18]:
rows_before = len(df)
valid_length = df["length_km"].between(MIN_LENGTH_KM, MAX_LENGTH_KM)
df = df[valid_length].reset_index(drop=True)
print(f"Видалено рядків із некоректною довжиною: {rows_before - len(df)}")

df.loc[df["price_eur"] < 0, "price_eur"] = np.nan
median_price = df.groupby("type")["price_eur"].transform("median")
df["price_eur"] = df["price_eur"].fillna(median_price).round(2)

df["description"] = df["description"].fillna("Опис відсутній")
print("Залишилось пропусків у price_eur:", df["price_eur"].isna().sum())

Видалено рядків із некоректною довжиною: 12
Залишилось пропусків у price_eur: 0


### 3.6. Перевірка оцінок (`ratings`)

Оцінки записані рядком через пробіл. Розбираємо їх у список чисел і перевіряємо, що кожна лежить у межах 1–10 (`MIN_RATING`, `MAX_RATING` з config.py).

In [19]:
MIN_RATING, MAX_RATING = 1, 10

def parse_ratings(raw):
    if pd.isna(raw):
        return np.nan
    return [int(part) for part in str(raw).split()]

def ratings_valid(values):
    return isinstance(values, list) and all(MIN_RATING <= v <= MAX_RATING for v in values)

df["ratings_list"] = df["ratings"].apply(parse_ratings)
has_ratings = df["ratings_list"].apply(lambda v: isinstance(v, list))
invalid_mask = has_ratings & ~df["ratings_list"].apply(ratings_valid)
print("Записів з оцінками поза діапазоном 1–10:", invalid_mask.sum())
df[invalid_mask][["route_id", "name", "ratings"]]

Записів з оцінками поза діапазоном 1–10: 4


,route_id,name,ratings
24,149,Велопетля Дністра (Буковель),9 15 8
132,145,Веломаршрут Карпат (Церматт),9 15 8
192,151,Прогулянка Монблану (Ставангер),9 15 8
275,258,Стежка Дністра (Вроцлав),9 15 8


In [20]:
# Некоректні відгуки не можна виправити, тому вважаємо, що відгуків немає
df.loc[invalid_mask, ["ratings", "ratings_list"]] = np.nan
print("Маршрутів без валідних відгуків:", df["ratings_list"].isna().sum())

Маршрутів без валідних відгуків: 12


## 4. Приведення типів

In [21]:
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")
print("Некоректних дат (стали NaT):", df["date_added"].isna().sum())

for column in ["type", "country", "difficulty", "bike_type"]:
    df[column] = df[column].astype("category")

# Цілі значення з пропусками — nullable-тип Int64
df["elevation_gain_m"] = df["elevation_gain_m"].astype("Int64")
df["stops_count"] = df["stops_count"].astype("Int64")

Некоректних дат (стали NaT): 2


Перевірка результату:

In [22]:
df.dtypes

route_id                     int64
type                      category
name                           str
country                   category
start_city                     str
length_km                  float64
description                    str
ratings                        str
difficulty                category
elevation_gain_m             Int64
bike_type                 category
stops_count                  Int64
date_added          datetime64[us]
price_eur                  float64
ratings_list                object
dtype: object

In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 288 entries, 0 to 287
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   route_id          288 non-null    int64         
 1   type              288 non-null    category      
 2   name              288 non-null    str           
 3   country           288 non-null    category      
 4   start_city        288 non-null    str           
 5   length_km         288 non-null    float64       
 6   description       288 non-null    str           
 7   ratings           276 non-null    str           
 8   difficulty        109 non-null    category      
 9   elevation_gain_m  109 non-null    Int64         
 10  bike_type         73 non-null     category      
 11  stops_count       52 non-null     Int64         
 12  date_added        286 non-null    datetime64[us]
 13  price_eur         288 non-null    float64       
 14  ratings_list      276 non-null    obj

## 5. Створення нових стовпців

Додаємо похідні показники. Розрахунок тривалості повторює логіку `estimated_duration_hours()` із класів проєкту (поліморфізм): різні типи маршрутів мають різну швидкість і додаткові поправки.

In [24]:
# 5.1. Середній бал та кількість відгуків
df["avg_rating"] = df["ratings_list"].apply(lambda v: np.mean(v) if isinstance(v, list) else np.nan).round(2)
df["ratings_count"] = df["ratings_list"].apply(lambda v: len(v) if isinstance(v, list) else 0)

# 5.2. Орієнтовна тривалість (години) — правила з models.py (основного коду)
speed = df["type"].astype(str).map({"Route": 5.0, "HikingRoute": 4.0, "CyclingRoute": 15.0, "BusTour": 40.0})
speed = speed.where(~((df["type"] == "CyclingRoute") & (df["bike_type"] == "гірський")), 10.0)

df["duration_hours"] = df["length_km"] / speed
df["duration_hours"] += (df["elevation_gain_m"].fillna(0).astype(float) / 600)   # HikingRoute
df["duration_hours"] += (df["stops_count"].fillna(0).astype(float) * 0.25)       # BusTour
df["duration_hours"] = df["duration_hours"].round(2)

# 5.3. Ціна за кілометр
df["price_per_km"] = (df["price_eur"] / df["length_km"]).round(2)

# 5.4. Категорія за довжиною
df["length_class"] = pd.cut(
    df["length_km"], bins=[0, 20, 100, 300, MAX_LENGTH_KM],
    labels=["коротка", "середня", "довга", "дуже довга"],
)

# 5.5. Рік додавання та автоматично згенерований показник (NumPy)
df["year_added"] = df["date_added"].dt.year.astype("Int64")
rng = np.random.default_rng(42)
df["popularity_score"] = rng.integers(50, 101, size=len(df))   # умовний бал популярності 50–100

df[["route_id", "type", "length_km", "avg_rating", "ratings_count", "duration_hours",
    "price_per_km", "length_class", "year_added", "popularity_score"]].head()

,route_id,type,length_km,avg_rating,ratings_count,duration_hours,price_per_km,length_class,year_added,popularity_score
0,229,BusTour,282.6,8.00,3,7.82,0.40,довга,2025,54
1,10,CyclingRoute,191.4,6.71,7,19.14,0.23,довга,2023,89
2,58,CyclingRoute,293.4,NaN,0,29.34,0.85,довга,2025,83
3,61,BusTour,423.3,8.50,4,13.08,0.82,дуже довга,2024,72
4,26,HikingRoute,6.3,7.14,7,5.33,48.24,коротка,2025,72


### 5.1. Перевірка розрахунків на класах проєкту

Щоб переконатися, що pandas-розрахунки збігаються з методами класів `Route`, створимо ті самі маршрути з `sample_data.py`, порахуємо для них середній бал і тривалість двома способами та порівняємо. (Блок працює, якщо пакет `tourist_routes` лежить на рівень вище; у Colab він пропускається.)

In [25]:
try:
    from tourist_routes.core.sample_data import create_sample_routes
except ImportError:
    print("Пакет tourist_routes недоступний — блок пропущено.")
else:
    routes = create_sample_routes()
    check = pd.DataFrame(route.to_dict() for route in routes)
    check["avg_pandas"] = check["ratings"].apply(np.mean).round(4)
    check["avg_classes"] = [round(r.average_rating(), 4) for r in routes]
    check["hours_classes"] = [round(r.estimated_duration_hours(), 2) for r in routes]

    speed_c = check["type"].map({"Route": 5.0, "HikingRoute": 4.0, "CyclingRoute": 15.0, "BusTour": 40.0})
    speed_c = speed_c.where(~((check["type"] == "CyclingRoute") & (check.get("bike_type") == "гірський")), 10.0)
    check["hours_pandas"] = (
        check["length_km"] / speed_c
        + check["elevation_gain_m"].fillna(0) / 600
        + check["stops_count"].fillna(0) * 0.25
    ).round(2)

    print("Середній бал збігається:", (check["avg_pandas"] == check["avg_classes"]).all())
    print("Тривалість збігається:  ", np.allclose(check["hours_pandas"], check["hours_classes"]))
    display(check[["name", "type", "avg_pandas", "avg_classes", "hours_pandas", "hours_classes"]])

Середній бал збігається: True
Тривалість збігається:   True


,name,type,avg_pandas,avg_classes,hours_pandas,hours_classes
0,Карпатська мандрівка,HikingRoute,9.0000,9.0000,14.25,14.25
1,Тропа Інків,HikingRoute,9.6667,9.6667,14.58,14.58
2,Веломаршрут вздовж Дунаю,CyclingRoute,9.0000,9.0000,21.33,21.33
3,Гірський трейл Татр,CyclingRoute,8.5000,8.5000,8.50,8.50
4,Тур довкола Монблану,BusTour,9.3333,9.3333,6.25,6.25
5,Прогулянка старим Львовом,Route,8.6667,8.6667,1.00,1.00


## 6. Дослідницький аналіз даних

### 6.1. Описова статистика

In [26]:
df.describe()

,route_id,length_km,elevation_gain_m,stops_count,date_added,price_eur,avg_rating,ratings_count,duration_hours,price_per_km,year_added,popularity_score
count,288.000000,288.000000,109.0,52.0,286,288.000000,276.000000,288.000000,288.000000,288.000000,286.0,288.000000
mean,150.767361,143.423958,1436.449541,7.346154,2024-06-24 20:53:42.377622,258.292535,7.915725,4.288194,11.017604,6.248368,2023.986014,74.923611
min,1.000000,2.000000,135.0,2.0,2023-01-01 00:00:00,1.900000,5.500000,0.000000,0.400000,0.070000,2023.0,50.000000
25%,75.750000,16.375000,748.0,4.0,2023-09-22 00:00:00,79.655000,7.237500,3.000000,4.845000,1.067500,2023.0,64.000000
50%,151.500000,56.450000,1450.0,7.0,2024-06-02 12:00:00,264.275000,8.000000,4.000000,10.550000,2.720000,2024.0,75.000000
75%,226.250000,200.650000,1980.0,10.25,2025-03-31 00:00:00,415.255000,8.670000,6.000000,16.555000,7.257500,2025.0,86.250000
max,300.000000,866.700000,2978.0,14.0,2026-01-04 00:00:00,599.320000,10.000000,7.000000,34.740000,87.420000,2026.0,100.000000
std,86.949210,194.435239,794.196676,3.813725,NaN,188.185392,0.979693,1.876946,7.132628,10.327239,0.841769,13.897374


In [27]:
numeric_columns = ["length_km", "price_eur", "avg_rating", "duration_hours"]
pd.DataFrame({
    "середнє": df[numeric_columns].mean(),
    "медіана": df[numeric_columns].median(),
    "мінімум": df[numeric_columns].min(),
    "максимум": df[numeric_columns].max(),
}).round(2)

,середнє,медіана,мінімум,максимум
length_km,143.42,56.45,2.0,866.70
price_eur,258.29,264.27,1.9,599.32
avg_rating,7.92,8.00,5.5,10.00
duration_hours,11.02,10.55,0.4,34.74


Середнє довжини помітно більше за медіану — розподіл зміщений праворуч через довгі автобусні тури.

### 6.2. Розподіл категоріальних даних

In [28]:
print("Типи маршрутів:")
display(df["type"].value_counts())
print("Країни:")
display(df["country"].value_counts())

Типи маршрутів:


type
HikingRoute     109
CyclingRoute     73
Route            54
BusTour          52
Name: count, dtype: int64

Країни:


country
Швейцарія     39
Австрія       35
Україна       35
Франція       35
Норвегія      32
Польща        31
Італія        29
Словаччина    27
Перу          25
Name: count, dtype: int64

In [29]:
print("Складність (пішохідні):")
display(df["difficulty"].value_counts())
print("Типи велосипедів:")
display(df["bike_type"].value_counts())
print("Категорії за довжиною:")
display(df["length_class"].value_counts())

Складність (пішохідні):


difficulty
складний    40
легкий      39
середній    30
Name: count, dtype: int64

Типи велосипедів:


bike_type
гірський    32
міський     23
шосейний    18
Name: count, dtype: int64

Категорії за довжиною:


length_class
середня       106
коротка        78
довга          65
дуже довга     39
Name: count, dtype: int64

In [30]:
country_counts = df["country"].value_counts()
type_counts = df["type"].value_counts()
print(f"Найпоширеніша країна: {country_counts.idxmax()} ({country_counts.max()})")
print(f"Найменш поширена країна: {country_counts.idxmin()} ({country_counts.min()})")
print(f"Найпоширеніший тип: {type_counts.idxmax()} ({type_counts.max()})")
print(f"Найменш поширений тип: {type_counts.idxmin()} ({type_counts.min()})")

Найпоширеніша країна: Швейцарія (39)
Найменш поширена країна: Перу (25)
Найпоширеніший тип: HikingRoute (109)
Найменш поширений тип: BusTour (52)


## 7. Фільтрація та сортування

### 7.1. Фільтрація за одним критерієм

In [31]:
# Маршрути в Україні
df[df["country"] == "Україна"][["route_id", "type", "name", "length_km", "avg_rating"]].head(10)

,route_id,type,name,length_km,avg_rating
3,61,BusTour,Екскурсійний тур Карпат (Одеса),423.3,8.50
24,149,CyclingRoute,Велопетля Дністра (Буковель),46.3,NaN
28,240,HikingRoute,Піший похід Доломітів (Ворохта),47.2,6.75
30,83,Route,Прогулянка Фйордів (Яремче),12.7,7.00
41,125,HikingRoute,Трек через Доломітів (Одеса),28.6,8.14
51,126,CyclingRoute,Трейл Доломітів (Київ),304.2,8.00
57,226,Route,Прогулянка Озерного краю (Одеса),5.2,6.33
60,112,HikingRoute,Трек через Озерного краю (Львів),5.9,8.67
75,120,BusTour,Автотур Фйордів (Одеса),292.2,8.00
79,138,CyclingRoute,Велотур Карпат (Київ),96.9,8.50


### 7.2. Фільтрація за декількома умовами

In [32]:
# Умова І (&): складні пішохідні маршрути з середнім балом не нижче 8
hard_hikes = df[(df["type"] == "HikingRoute") & (df["difficulty"] == "складний") & (df["avg_rating"] >= 8)]
print("Складні пішохідні маршрути з балом ≥ 8:", len(hard_hikes))
hard_hikes[["route_id", "name", "country", "length_km", "elevation_gain_m", "avg_rating"]].head(10)

Складні пішохідні маршрути з балом ≥ 8: 17


,route_id,name,country,length_km,elevation_gain_m,avg_rating
27,292,Піший похід Доломітів (Кошице),Словаччина,54.5,329,8.40
36,173,Трек через Монблану (Люцерн),Швейцарія,53.7,1906,8.17
50,85,Хребтами Старого міста (Верона),Італія,64.4,753,8.25
66,60,Піший похід Старого міста (Люцерн),Швейцарія,54.0,1446,8.00
69,184,Піший похід Озерного краю (Берген),Норвегія,50.8,1455,8.20
72,272,Хребтами Доломітів (Братислава),Словаччина,43.4,2584,8.00
110,239,Трек через Озерного краю (Церматт),Швейцарія,67.7,1041,8.33
120,155,Стежка Доломітів (Інтерлакен),Швейцарія,51.5,1069,9.00
126,140,Піший похід Фйордів (Арекіпа),Перу,79.2,1313,8.50
127,196,Хребтами Доломітів (Куско),Перу,28.4,1721,8.50


In [33]:
# Умова АБО (|): дешеві (до 50 €) або дуже короткі (до 10 км) маршрути
cheap_or_short = df[(df["price_eur"] < 50) | (df["length_km"] < 10)]
print("Дешеві або короткі маршрути:", len(cheap_or_short))
cheap_or_short[["route_id", "type", "length_km", "price_eur"]].head(10)

Дешеві або короткі маршрути: 72


,route_id,type,length_km,price_eur
1,10,CyclingRoute,191.4,44.56
4,26,HikingRoute,6.3,303.91
13,174,Route,10.6,12.71
14,293,Route,14.8,8.11
15,6,Route,12.1,25.48
16,232,CyclingRoute,152.1,47.48
18,297,Route,4.9,36.74
30,83,Route,12.7,30.07
32,252,Route,3.1,39.62
33,198,Route,7.0,25.98


In [34]:
# Комбінація: велосипедні маршрути довжиною 100–250 км у Австрії або Франції
df[(df["type"] == "CyclingRoute")
   & df["length_km"].between(100, 250)
   & df["country"].isin(["Австрія", "Франція"])][["route_id", "name", "country", "length_km", "bike_type"]].head(10)

,route_id,name,country,length_km,bike_type
1,10,Веломаршрут Фйордів (Шамоні),Франція,191.4,гірський
23,144,Велотур Фйордів (Відень),Австрія,198.8,міський
26,278,Трейл Татр (Страсбург),Франція,130.4,гірський
64,79,Велопетля Фйордів (Страсбург),Франція,102.9,міський
122,222,Веломаршрут Фйордів (Страсбург),Франція,180.7,гірський
226,237,Трейл Доломітів (Шамоні),Франція,113.7,міський
231,2,Велотур Балтики (Ліон),Франція,249.4,шосейний
244,260,Веломаршрут Дунаю (Пассау),Австрія,219.1,міський
249,35,Велопетля Озерного краю (Інсбрук),Австрія,220.6,міський
271,22,Велотур Дунаю (Пассау),Австрія,186.7,шосейний


### 7.3. Сортування

In [35]:
# За одним стовпцем: найвищий середній бал
df.sort_values("avg_rating", ascending=False)[["route_id", "name", "type", "avg_rating"]].head(10)

,route_id,name,type,avg_rating
246,54,Автотур Доломітів (Зальцбург),BusTour,10.00
122,222,Веломаршрут Фйордів (Страсбург),CyclingRoute,10.00
284,72,Веломаршрут Фйордів (Ставангер),CyclingRoute,9.75
237,276,Велопетля Карпат (Кортіна),CyclingRoute,9.71
84,119,Веломаршрут Озерного краю (Пассау),CyclingRoute,9.67
257,274,Веломаршрут Фйордів (Пассау),CyclingRoute,9.67
99,238,Екскурсійний тур Доломітів (Яремче),BusTour,9.50
191,62,Велотур Карпат (Люцерн),CyclingRoute,9.50
139,187,Трейл Дністра (Куско),CyclingRoute,9.50
178,300,Автотур Карпат (Церматт),BusTour,9.50


In [36]:
# За декількома стовпцями: тип (А→Я), далі довжина (за спаданням)
df.sort_values(["type", "length_km"], ascending=[True, False])[["route_id", "type", "name", "length_km"]].head(10)

,route_id,type,name,length_km
229,40,BusTour,Автобусом через Фйордів (Шамоні),866.7
49,105,BusTour,Автобусом через Альп (Інсбрук),822.7
282,122,BusTour,Автотур Монблану (Ніцца),818.1
102,156,BusTour,Автобусом через Татр (Закопане),812.4
261,55,BusTour,Екскурсійний тур Карпат (Берген),805.6
178,300,BusTour,Автотур Карпат (Церматт),797.2
283,21,BusTour,Автобусом через Старого міста (Інтерлакен),790.8
159,211,BusTour,Автобусом через Анд (Рим),790.3
181,1,BusTour,Автотур Балтики (Рим),783.2
156,256,BusTour,Автобусом через Монблану (Закопане),753.7


## 8. Групування та агрегація

### 8.1. Групування за однією змінною

In [37]:
by_type = df.groupby("type", observed=True)["length_km"].agg(["count", "sum", "mean", "min", "max"]).round(2)
by_type

,count,sum,mean,min,max
type,,,,,
BusTour,52,23405.5,450.11,80.8,866.7
CyclingRoute,73,13064.9,178.97,33.9,347.4
HikingRoute,109,4352.3,39.93,5.3,79.6
Route,54,483.4,8.95,2.0,14.8


In [38]:
by_country = (
    df.groupby("country", observed=True)
      .agg(routes=("route_id", "count"),
           total_km=("length_km", "sum"),
           avg_length_km=("length_km", "mean"),
           min_length_km=("length_km", "min"),
           max_length_km=("length_km", "max"),
           avg_rating=("avg_rating", "mean"),
           avg_price_eur=("price_eur", "mean"))
      .round(2)
      .sort_values("routes", ascending=False)
)
by_country

,routes,total_km,avg_length_km,min_length_km,max_length_km,avg_rating,avg_price_eur
country,,,,,,,
Швейцарія,39,7034.2,180.36,4.9,797.2,8.20,270.28
Україна,35,5402.7,154.36,3.2,706.4,7.72,274.21
Австрія,35,4971.8,142.05,3.6,822.7,8.19,290.58
Франція,35,4651.8,132.91,3.3,866.7,7.54,207.05
Норвегія,32,3876.1,121.13,3.6,805.6,7.95,202.09
Польща,31,4155.0,134.03,2.0,812.4,7.67,271.15
Італія,29,4319.5,148.95,2.9,790.3,7.67,221.37
Словаччина,27,3827.6,141.76,7.0,742.1,8.17,307.44
Перу,25,3067.4,122.70,2.6,452.5,8.13,289.59


### 8.2. Групування за двома категоріями

In [39]:
by_country_type = (
    df.groupby(["country", "type"], observed=True)
      .agg(routes=("route_id", "count"),
           avg_length_km=("length_km", "mean"),
           avg_rating=("avg_rating", "mean"),
           avg_price_eur=("price_eur", "mean"))
      .round(2)
      .reset_index()
)
by_country_type.head(15)

,country,type,routes,avg_length_km,avg_rating,avg_price_eur
0,Італія,BusTour,6,520.87,7.42,357.33
1,Італія,CyclingRoute,4,173.65,7.43,282.60
2,Італія,HikingRoute,10,42.99,7.80,296.93
3,Італія,Route,9,7.76,7.84,19.57
4,Австрія,BusTour,4,406.15,8.73,425.02
5,Австрія,CyclingRoute,13,212.10,8.26,340.37
6,Австрія,HikingRoute,12,44.31,7.98,325.94
7,Австрія,Route,6,9.70,8.13,22.36
8,Норвегія,BusTour,5,470.24,8.00,283.99
9,Норвегія,CyclingRoute,5,184.46,8.85,280.60


In [40]:
# Зведена таблиця: кількість маршрутів за країною і типом
pd.pivot_table(df, index="country", columns="type", values="route_id", aggfunc="count", fill_value=0, observed=True)

type,BusTour,CyclingRoute,HikingRoute,Route
country,,,,
Італія,6,4,10,9
Австрія,4,13,12,6
Норвегія,5,5,16,6
Перу,6,7,9,3
Польща,7,6,11,7
Словаччина,5,6,11,5
Україна,8,9,13,5
Франція,5,9,13,8
Швейцарія,6,14,14,5


In [41]:
# Які типи мають найвищу середню оцінку та ціну за кілометр
type_summary = df.groupby("type", observed=True)[["avg_rating", "price_per_km", "duration_hours"]].mean().round(2)
type_summary.sort_values("avg_rating", ascending=False)

,avg_rating,price_per_km,duration_hours
type,,,
CyclingRoute,8.15,2.57,14.34
BusTour,8.07,1.20,13.09
HikingRoute,7.79,12.51,12.38
Route,7.70,3.44,1.79


## 9. Збереження результатів

In [42]:
# Очищений датасет (службовий стовпець зі списками оцінок не зберігаємо)
processed = df.drop(columns=["ratings_list"])
processed.to_csv(OUT_DIR / "processed_routes.csv", index=False, encoding="utf-8")

# Результат групування за країною та типом
by_country_type.to_csv(OUT_DIR / "grouped_country_type.csv", index=False, encoding="utf-8")

print("Збережено:", OUT_DIR / "processed_routes.csv", "та", OUT_DIR / "grouped_country_type.csv")

Збережено: ../data/processed/processed_routes.csv та ../data/processed/grouped_country_type.csv


Перевірка: читаємо збережені файли назад і порівнюємо з даними в пам'яті.

In [43]:
check_processed = pd.read_csv(OUT_DIR / "processed_routes.csv", parse_dates=["date_added"])
check_grouped = pd.read_csv(OUT_DIR / "grouped_country_type.csv")

print("processed_routes.csv:", check_processed.shape, "| у пам'яті:", processed.shape)
print("grouped_country_type.csv:", check_grouped.shape, "| у пам'яті:", by_country_type.shape)
print("Розміри збігаються:", check_processed.shape == processed.shape and check_grouped.shape == by_country_type.shape)
print("Пропусків у ключових стовпцях:", check_processed[["route_id", "type", "country", "length_km", "price_eur"]].isna().sum().sum())
check_processed.head()

processed_routes.csv: (288, 21) | у пам'яті: (288, 21)
grouped_country_type.csv: (36, 6) | у пам'яті: (36, 6)
Розміри збігаються: True
Пропусків у ключових стовпцях: 0


,route_id,type,name,country,start_city,length_km,description,ratings,difficulty,elevation_gain_m,bike_type,stops_count,date_added,price_eur,avg_rating,ratings_count,duration_hours,price_per_km,length_class,year_added,popularity_score
0,229,BusTour,Екскурсійний тур Балтики (Пассау),Австрія,Пассау,282.6,Маршрут проходить через національний парк.,8 8 8,NaN,NaN,NaN,3.0,2025-08-17,111.95,8.00,3,7.82,0.40,довга,2025.0,54
1,10,CyclingRoute,Веломаршрут Фйордів (Шамоні),Франція,Шамоні,191.4,Мальовничий маршрут з чудовими краєвидами.,5 6 7 8 9 5 7,NaN,NaN,гірський,NaN,2023-03-19,44.56,6.71,7,19.14,0.23,довга,2023.0,89
2,58,CyclingRoute,Веломаршрут Доломітів (Відень),Австрія,Відень,293.4,Мальовничий маршрут з чудовими краєвидами.,NaN,NaN,NaN,гірський,NaN,2025-06-24,248.80,NaN,0,29.34,0.85,довга,2025.0,83
3,61,BusTour,Екскурсійний тур Карпат (Одеса),Україна,Одеса,423.3,Є кемпінги та місця для відпочинку по дорозі.,9 9 9 7,NaN,NaN,NaN,10.0,2024-09-09,347.18,8.50,4,13.08,0.82,дуже довга,2024.0,72
4,26,HikingRoute,Стежка Карпат (Пассау),Австрія,Пассау,6.3,Опис відсутній,8 7 7 8 6 6 8,складний,2252.0,NaN,NaN,2025-11-21,303.91,7.14,7,5.33,48.24,коротка,2025.0,72


In [44]:
check_grouped.head()

,country,type,routes,avg_length_km,avg_rating,avg_price_eur
0,Італія,BusTour,6,520.87,7.42,357.33
1,Італія,CyclingRoute,4,173.65,7.43,282.60
2,Італія,HikingRoute,10,42.99,7.80,296.93
3,Італія,Route,9,7.76,7.84,19.57
4,Австрія,BusTour,4,406.15,8.73,425.02


## 10. Висновки

У роботі було:

1. Зчитано CSV-датасет туристичних маршрутів (312 рядків, 14 стовпців) і виконано первинний огляд: `head`, `tail`, `info`, `dtypes`, `nunique`.
2. Очищено дані: видалено повні дублікати, виправлено регістр і пробіли у текстових полях, замінено кому на крапку в числах, видалено рядки з неможливою довжиною, заповнено пропуски ціни медіаною за типом маршруту, відхилено оцінки поза діапазоном 1–10. Структурні пропуски (поля, що існують лише для певного типу маршруту) свідомо залишено.
3. Приведено типи: числові, `category` для категоріальних стовпців, `datetime` для дат, `Int64` для цілих із пропусками.
4. Створено нові стовпці (середній бал, тривалість за правилами з models.py, ціна за км, категорія довжини, випадковий бал популярності) і перевірено розрахунки на маршрутах із `sample_data.py` основного коду.
5. Виконано описову статистику, аналіз розподілу категорій, фільтрацію (`&`, `|`, `isin`, `between`), сортування й групування за однією та двома змінними.
6. Збережено очищений датасет `processed_routes.csv` та результат групування `grouped_country_type.csv` і перевірено, що вони коректно відкриваються.